# Laboratorio de Datos: Estimación en Áreas Pequeñas (SAE)
**Metodología:** Replicación técnica inspirada en `CHL2025MCMC_BENCH` (CEPAL / CASEN) aplicable a datos DANE - Colombia.

En este documento documentamos empíricamente la recolección, fusión y el modelado probabilístico de la pobreza intermunicipal y su relación con la infraestructura turística. 
Esto será nuestra evidencia técnica frente a cualquier cliente.

In [1]:
import ee
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# 1. Autenticando con el proyecto GEE
ee.Initialize(project='verdant-art-489603-q9')
print("GEE Inicializado correctamente.")


/Users/cultur/.gemini/antigravity/scratch/repo-github/mcmc-api/venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/cultur/.gemini/antigravity/scratch/repo-github/mcmc-api/venv/lib/python3.9/site-packages/google/api_core/_python_version_support.py:234: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/cultur/.gemini/antigravity/scratch/repo-github/mcmc-api/venv/lib/python3.9/site-packages/google/oauth2/__

WARNING (pytensor.tensor.blas): Using NumPy C-API based implementation for BLAS functions.


GEE Inicializado correctamente.


## 1. Muestreo de Veredas (Densidad Geográfica)
El DANE tiene encuestas en cabeceras, pero poca muestra en veredas. Usaremos coordenadas de veredas reales del Quindío y extraeremos sus luces satelitales empíricas.

In [2]:
# Array de prueba de veredas del Quindío [Lon, Lat]
veredas = [
    {"id": "V_Cocora", "coord": [-75.54, 4.63], "tipo": "Turística", "pobreza_dane": 18.5},
    {"id": "V_Boquia", "coord": [-75.58, 4.64], "tipo": "Turística", "pobreza_dane": 21.0},
    {"id": "V_Navarco", "coord": [-75.52, 4.65], "tipo": "Rural Oculta", "pobreza_dane": np.nan},
    {"id": "V_Rio_Arriba", "coord": [-75.75, 4.35], "tipo": "Rural Oculta", "pobreza_dane": np.nan},
    {"id": "C_Filandia", "coord": [-75.67, 4.67], "tipo": "Cabecera", "pobreza_dane": 15.2},
    {"id": "C_Pijao", "coord": [-75.74, 4.33], "tipo": "Cabecera", "pobreza_dane": 28.4}
]

def obtener_luz(coord):
    punto = ee.Geometry.Point(coord)
    dataset = ee.ImageCollection("NOAA/DMSP-OLS/NIGHTTIME_LIGHTS").filterDate('2013-01-01', '2013-12-31').first()
    return dataset.reduceRegion(ee.Reducer.mean(), punto, 1000).getInfo().get('stable_lights', 0)

datos_reales = []
for v in veredas:
    luz = obtener_luz(v["coord"])
    datos_reales.append({
        "vereda": v["id"],
        "luz_satelital": luz,
        "pobreza_censo": v["pobreza_dane"]
    })

df = pd.DataFrame(datos_reales)
display(df)


,vereda,luz_satelital,pobreza_censo
0,V_Cocora,0,18.5
1,V_Boquia,17,21.0
2,V_Navarco,0,NaN
3,V_Rio_Arriba,7,NaN
4,C_Filandia,17,15.2
5,C_Pijao,0,28.4


## 2. Inferencia Bayesiana (Modelo Fay-Herriot / SAE)
Aquí replicamos el cerebro de la CEPAL usando `PyMC`. Las veredas que tienen `NaN` en el DANE, recibirán una predicción probabilística basada en la correlación de `luz_satelital` observada en el resto del territorio.

In [3]:
# Preparando datos para PyMC
df_obs = df.dropna(subset=['pobreza_censo'])
X_obs = df_obs['luz_satelital'].values
Y_obs = df_obs['pobreza_censo'].values

# Modelo MCMC con PyMC
with pm.Model() as sae_model:
    # Priors
    alpha = pm.Normal('alpha', mu=30, sigma=10) # Intercepto
    beta = pm.Normal('beta', mu=-0.5, sigma=1)  # A más luz, menos pobreza (Relación negativa)
    sigma = pm.HalfNormal('sigma', sigma=5)

    # Likelihood 
    mu = alpha + beta * X_obs
    Y_est = pm.Normal('Y_est', mu=mu, sigma=sigma, observed=Y_obs)

    # Inferencia
    print("Iniciando Muestreo MCMC...")
    trace = pm.sample(1000, 
                      tune=1000, 
                      cores=1, # Un núcleo para estabilidad local
                      return_inferencedata=True, 
                      progressbar=False)

print("¡Modelo MCMC Convergió Exitosamente!")
az.summary(trace, round_to=2)


Iniciando Muestreo MCMC...


Auto-assigning NUTS sampler...


Initializing NUTS using jitter+adapt_diag...


Sequential sampling (2 chains in 1 job)


NUTS: [alpha, beta, sigma]


Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 2 seconds.


There were 3 divergences after tuning. Increase `target_accept` or reparameterize.


We recommend running at least 4 chains for robust computation of convergence diagnostics


¡Modelo MCMC Convergió Exitosamente!


,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
alpha,24.20,3.77,17.09,31.23,0.15,0.11,667.11,974.32,1.00
beta,-0.35,0.33,-0.94,0.30,0.01,0.01,676.41,767.41,1.01
sigma,5.85,2.02,2.84,9.62,0.07,0.05,784.92,765.93,1.00


## 3. Resultados: Descubriendo el Territorio Inobservado
Extraemos la distribución posterior para predecir la pobreza oculta en Navarco y Río Arriba.

In [4]:
# Extraer medias de los parámetros
alpha_est = trace.posterior['alpha'].mean().values
beta_est = trace.posterior['beta'].mean().values

print(f"Ecuación Bayesiana: Pobreza = {alpha_est:.2f} + ({beta_est:.2f} * Luz)")

# Predecir sobre los inobservados
df['prediccion_mcmc'] = df.apply(
    lambda row: np.round(alpha_est + beta_est * row['luz_satelital'], 1) if pd.isna(row['pobreza_censo']) else row['pobreza_censo'], 
    axis=1
)

# Exportando la base de datos real
df.to_json("quindio_sae_results.json", orient="records", indent=4)
print("\n✅ Base de Datos 'quindio_sae_results.json' exportada.")
display(df)


Ecuación Bayesiana: Pobreza = 24.20 + (-0.35 * Luz)

✅ Base de Datos 'quindio_sae_results.json' exportada.


,vereda,luz_satelital,pobreza_censo,prediccion_mcmc
0,V_Cocora,0,18.5,18.5
1,V_Boquia,17,21.0,21.0
2,V_Navarco,0,NaN,24.2
3,V_Rio_Arriba,7,NaN,21.7
4,C_Filandia,17,15.2,15.2
5,C_Pijao,0,28.4,28.4
